# Issue 3: Credits Data Cleaning

This notebook cleans `credits.csv` and extracts useful cast and crew information.

## Goals

- Inspect the dataset for missing values and duplicates.
- Parse the `cast` and `crew` columns.
- Extract each movie's director and top five cast members.
- Calculate cast size and crew size.
- Retain one valid row per movie ID.
- Save the cleaned output to `data/processed/credits.csv`.

In [18]:
from pathlib import Path
import pandas as pd

# Find the project folder.
project_root = Path.cwd()

# This handles cases where the notebook starts inside notebooks/preprocessing.
if not (project_root / "data" / "raw" / "credits.csv").exists():
    project_root = project_root.parent.parent

credits_path = project_root / "data" / "raw" / "credits.csv"

print("Project folder:", project_root)
print("Credits file:", credits_path)
print("File exists:", credits_path.exists())

Project folder: /Users/kevinphon/Desktop/DS-ML Project Fall Semester/css-dsml-fall2027
Credits file: /Users/kevinphon/Desktop/DS-ML Project Fall Semester/css-dsml-fall2027/data/raw/credits.csv
File exists: True


In [19]:
credits = pd.read_csv(credits_path)

print("Dataset dimensions:", credits.shape)
display(credits.head())

Dataset dimensions: (45476, 3)


,cast,crew,id
0,"[{'cast_id': 14, 'character': 'Woody (voice)',...","[{'credit_id': '52fe4284c3a36847f8024f49', 'de...",862
1,"[{'cast_id': 1, 'character': 'Alan Parrish', '...","[{'credit_id': '52fe44bfc3a36847f80a7cd1', 'de...",8844
2,"[{'cast_id': 2, 'character': 'Max Goldman', 'c...","[{'credit_id': '52fe466a9251416c75077a89', 'de...",15602
3,"[{'cast_id': 1, 'character': ""Savannah 'Vannah...","[{'credit_id': '52fe44779251416c91011acb', 'de...",31357
4,"[{'cast_id': 1, 'character': 'George Banks', '...","[{'credit_id': '52fe44959251416c75039ed7', 'de...",11862


In [20]:
print("Number of rows:", credits.shape[0])
print("Number of columns:", credits.shape[1])

display(credits.dtypes.to_frame(name="data_type"))

Number of rows: 45476
Number of columns: 3


,data_type
cast,str
crew,str
id,int64


In [21]:
missing_values = credits.isna().sum().to_frame(name="missing_values")

print("Total duplicate rows:", credits.duplicated().sum())
print("Missing movie IDs:", credits["id"].isna().sum())
print("Duplicated movie IDs:", credits["id"].duplicated().sum())

display(missing_values)

Total duplicate rows: 37
Missing movie IDs: 0
Duplicated movie IDs: 44


,missing_values
cast,0
crew,0
id,0


### Duplicate Inspection

The dataset contains 37 completely duplicated rows and 44 duplicated movie ID occurrences. Because the duplicated-ID count is larger than the complete-row duplicate count, some movies may have conflicting cast or crew records. We inspect these records before deciding how to clean them.

In [22]:
duplicate_id_rows = (
    credits[credits["id"].duplicated(keep=False)]
    .sort_values("id")
)

print("Rows involving duplicated IDs:", len(duplicate_id_rows))
print(
    "Number of unique IDs that appear more than once:",
    duplicate_id_rows["id"].nunique()
)

display(duplicate_id_rows)

Rows involving duplicated IDs: 87
Number of unique IDs that appear more than once: 43


,cast,crew,id
25885,"[{'cast_id': 12, 'character': 'The Creature', ...","[{'credit_id': '52fe4380c3a36847f80590dd', 'de...",3057
25950,"[{'cast_id': 12, 'character': 'The Creature', ...","[{'credit_id': '52fe4380c3a36847f80590dd', 'de...",3057
33838,"[{'cast_id': 15, 'character': 'Chuck Barris', ...","[{'credit_id': '52fe43e2c3a36847f80760a9', 'de...",4912
5865,"[{'cast_id': 15, 'character': 'Chuck Barris', ...","[{'credit_id': '52fe43e2c3a36847f80760b5', 'de...",4912
9165,"[{'cast_id': 11, 'character': 'Jef Costello', ...","[{'credit_id': '52fe440ac3a36847f807ee01', 'de...",5511
...,...,...,...
25887,"[{'cast_id': 7, 'character': 'Hollander', 'cre...","[{'credit_id': '52fe4da29251416c9111ce5d', 'de...",199591
24163,"[{'cast_id': 2, 'character': 'Ebba', 'credit_i...","[{'credit_id': '534fd1a80e0a267eb6000e32', 'de...",265189
45275,"[{'cast_id': 2, 'character': 'Ebba', 'credit_i...","[{'credit_id': '534fd1a80e0a267eb6000e32', 'de...",265189
33196,"[{'cast_id': 1, 'character': 'Jenjira', 'credi...","[{'credit_id': '5448c8efc3a3680fb4001582', 'de...",298721


### Conflicting Duplicate IDs

Most repeated IDs may be completely identical records. We first remove exact duplicate rows in a temporary copy, then inspect any movie IDs that still appear more than once. These remaining rows represent potentially conflicting cast or crew information.

In [23]:
# Create a temporary copy without completely identical rows.
credits_without_exact_duplicates = credits.drop_duplicates().copy()

# Find IDs that still appear more than once.
conflicting_id_rows = (
    credits_without_exact_duplicates[
        credits_without_exact_duplicates["id"].duplicated(keep=False)
    ]
    .sort_values("id")
)

print("Original rows:", len(credits))
print(
    "Rows after removing exact duplicates:",
    len(credits_without_exact_duplicates)
)
print("Remaining conflicting rows:", len(conflicting_id_rows))
print(
    "Number of IDs with different records:",
    conflicting_id_rows["id"].nunique()
)

display(conflicting_id_rows)

Original rows: 45476
Rows after removing exact duplicates: 45439
Remaining conflicting rows: 14
Number of IDs with different records: 7


,cast,crew,id
5865,"[{'cast_id': 15, 'character': 'Chuck Barris', ...","[{'credit_id': '52fe43e2c3a36847f80760b5', 'de...",4912
33838,"[{'cast_id': 15, 'character': 'Chuck Barris', ...","[{'credit_id': '52fe43e2c3a36847f80760a9', 'de...",4912
9576,"[{'cast_id': 1, 'character': 'Gerard Carriere'...","[{'credit_id': '5468acec22136e68c9000d53', 'de...",69234
26638,"[{'cast_id': 1, 'character': 'Gerard Carriere'...","[{'credit_id': '52fe47bfc3a368484e0d77bf', 'de...",69234
16167,"[{'cast_id': 1004, 'character': 'Luke Oarum', ...","[{'credit_id': '52fe4a269251416c750df623', 'de...",99080
38882,"[{'cast_id': 1004, 'character': 'Luke Oarum', ...","[{'credit_id': '52fe4a269251416c750df61d', 'de...",99080
4356,"[{'cast_id': 3, 'character': 'Camille Claudel'...","[{'credit_id': '52fe4ad6c3a36847f81e461b', 'de...",110428
23533,"[{'cast_id': 3, 'character': 'Camille Claudel'...","[{'credit_id': '577ed5389251416976004432', 'de...",110428
838,"[{'cast_id': 4, 'character': 'Mihoko Nakagawa'...","[{'credit_id': '52fe4b9ac3a368484e190d25', 'de...",132641
30013,"[{'cast_id': 4, 'character': 'Mihoko Nakagawa'...","[{'credit_id': '56365ed9925141285701b06e', 'de...",132641


## Parse Cast and Crew Columns

The `cast` and `crew` columns are stored as strings that resemble Python lists of dictionaries. We use `ast.literal_eval` to safely convert them into actual Python lists. Empty or invalid values are represented separately so they can be identified and handled.

In [24]:
import ast


def parse_list_column(value):
    """Convert a string representation of a list into a Python list."""
    if not isinstance(value, str) or not value.strip():
        return []

    try:
        parsed_value = ast.literal_eval(value)
    except (ValueError, SyntaxError):
        return None

    if isinstance(parsed_value, list):
        return parsed_value

    return None


credits_parsed = credits_without_exact_duplicates.copy()

credits_parsed["cast_parsed"] = credits_parsed["cast"].apply(
    parse_list_column
)
credits_parsed["crew_parsed"] = credits_parsed["crew"].apply(
    parse_list_column
)

print(
    "Cast parsing failures:",
    credits_parsed["cast_parsed"].isna().sum()
)
print(
    "Crew parsing failures:",
    credits_parsed["crew_parsed"].isna().sum()
)

print(
    "Movies with an empty cast list:",
    credits_parsed["cast_parsed"].apply(
        lambda value: isinstance(value, list) and len(value) == 0
    ).sum()
)

print(
    "Movies with an empty crew list:",
    credits_parsed["crew_parsed"].apply(
        lambda value: isinstance(value, list) and len(value) == 0
    ).sum()
)

Cast parsing failures: 0
Crew parsing failures: 0
Movies with an empty cast list: 2414
Movies with an empty crew list: 771


## Extract the Director

A movie's crew can contain many people with different jobs. We select the first person whose job is `Director`. If none is listed, the director remains missing.

In [25]:
def get_director(crew_members):
    if not isinstance(crew_members, list):
        return None

    for person in crew_members:
        if isinstance(person, dict) and person.get("job") == "Director":
            return person.get("name")

    return None


credits_parsed["director"] = credits_parsed["crew_parsed"].apply(get_director)

display(credits_parsed[["id", "director"]].head(10))

,id,director
0,862,John Lasseter
1,8844,Joe Johnston
2,15602,Howard Deutch
3,31357,Forest Whitaker
4,11862,Charles Shyer
5,949,Michael Mann
6,11860,Sydney Pollack
7,45325,Peter Hewitt
8,9091,Peter Hyams
9,710,Martin Campbell


## Extract the Top Five Cast Members

The `order` field records cast billing order. We sort by that field and keep up to five names. A movie with no cast receives an empty list.

In [26]:
def get_top_five_cast(cast_members):
    if not isinstance(cast_members, list):
        return []

    valid_members = [
        person for person in cast_members
        if isinstance(person, dict)
    ]

    ordered_members = sorted(
        valid_members,
        key=lambda person: (
            person.get("order")
            if isinstance(person.get("order"), int)
            else float("inf")
        )
    )

    names = [
        person["name"].strip()
        for person in ordered_members
        if isinstance(person.get("name"), str)
        and person["name"].strip()
    ]

    return names[:5]


credits_parsed["top_five_cast"] = credits_parsed["cast_parsed"].apply(
    get_top_five_cast
)

display(credits_parsed[["id", "top_five_cast"]].head(10))

,id,top_five_cast
0,862,"[Tom Hanks, Tim Allen, Don Rickles, Jim Varney..."
1,8844,"[Robin Williams, Jonathan Hyde, Kirsten Dunst,..."
2,15602,"[Walter Matthau, Jack Lemmon, Ann-Margret, Sop..."
3,31357,"[Whitney Houston, Angela Bassett, Loretta Devi..."
4,11862,"[Steve Martin, Diane Keaton, Martin Short, Kim..."
5,949,"[Al Pacino, Robert De Niro, Val Kilmer, Jon Vo..."
6,11860,"[Harrison Ford, Julia Ormond, Greg Kinnear, An..."
7,45325,"[Jonathan Taylor Thomas, Brad Renfro, Rachael ..."
8,9091,"[Jean-Claude Van Damme, Powers Boothe, Dorian ..."
9,710,"[Pierce Brosnan, Sean Bean, Izabella Scorupco,..."


## Count Cast and Crew Members (Cast size & Crew size)

Cast size and crew size count the people in each parsed list. An empty list has size zero.

In [27]:
credits_parsed["cast_size"] = credits_parsed["cast_parsed"].apply(
    lambda members: len(members) if isinstance(members, list) else 0
)

credits_parsed["crew_size"] = credits_parsed["crew_parsed"].apply(
    lambda members: len(members) if isinstance(members, list) else 0
)

display(
    credits_parsed[
        ["id", "director", "top_five_cast", "cast_size", "crew_size"]
    ].head(10)
)

,id,director,top_five_cast,cast_size,crew_size
0,862,John Lasseter,"[Tom Hanks, Tim Allen, Don Rickles, Jim Varney...",13,106
1,8844,Joe Johnston,"[Robin Williams, Jonathan Hyde, Kirsten Dunst,...",26,16
2,15602,Howard Deutch,"[Walter Matthau, Jack Lemmon, Ann-Margret, Sop...",7,4
3,31357,Forest Whitaker,"[Whitney Houston, Angela Bassett, Loretta Devi...",10,10
4,11862,Charles Shyer,"[Steve Martin, Diane Keaton, Martin Short, Kim...",12,7
5,949,Michael Mann,"[Al Pacino, Robert De Niro, Val Kilmer, Jon Vo...",65,71
6,11860,Sydney Pollack,"[Harrison Ford, Julia Ormond, Greg Kinnear, An...",57,53
7,45325,Peter Hewitt,"[Jonathan Taylor Thomas, Brad Renfro, Rachael ...",7,4
8,9091,Peter Hyams,"[Jean-Claude Van Damme, Powers Boothe, Dorian ...",6,9
9,710,Martin Campbell,"[Pierce Brosnan, Sean Bean, Izabella Scorupco,...",20,46


## Check Missing Cast, Crew, and Director Information

Some CSV cells contain an empty list (`[]`) rather than a missing value. We count these cases before deciding how to represent them in the cleaned data.

In [28]:
print("Movies with no director:", credits_parsed["director"].isna().sum())
print("Movies with no cast:", (credits_parsed["cast_size"] == 0).sum())
print("Movies with no crew:", (credits_parsed["crew_size"] == 0).sum())

display(
    credits_parsed.loc[
        (credits_parsed["cast_size"] == 0)
        | (credits_parsed["crew_size"] == 0),
        ["id", "director", "top_five_cast", "cast_size", "crew_size"]
    ].head(10)
)

Movies with no director: 887
Movies with no cast: 2414
Movies with no crew: 771


,id,director,top_five_cast,cast_size,crew_size
137,124639,Sandip Ray,[],0,1
189,56088,NaN,"[Kurtis Blow, Sean Combs, Snoop Dogg, Dr. Dre,...",8,0
240,43475,Art Clokey,[],0,1
393,42981,Theodore Thomas,[],0,1
438,24257,Bruce Brown,[],0,1
595,124472,Jean Bach,[],0,1
614,123505,NaN,"[Lisa Eichhorn, Stanley Tucci, Caroline Aaron,...",11,0
620,166901,Clément Virgo,[],0,1
635,339428,NaN,"[Erik Goertz, Anton Rattinger, Dieter Wardetzk...",7,0
661,318177,NaN,[],0,0


## Handle Missing Credit Information

We keep movies even when credits are incomplete. No cast is represented by an empty list (`[]`) and a cast size of `0`. No crew is represented by a crew size of `0` and a missing director. A movie can have crew members but still have no one labeled `Director`.

After removing exact duplicate rows, 2,414 movies have no cast, 771 have no crew, and 887 have no listed director. Keeping these movies preserves their IDs for later work with other movie data.

In [29]:
no_cast = credits_parsed["cast_size"] == 0
no_crew = credits_parsed["crew_size"] == 0

assert credits_parsed.loc[no_cast, "top_five_cast"].apply(
    lambda names: names == []
).all()

assert credits_parsed.loc[no_crew, "director"].isna().all()

assert credits_parsed["top_five_cast"].apply(
    lambda names: (
        isinstance(names, list)
        and len(names) <= 5
        and all(isinstance(name, str) for name in names)
    )
).all()

print("Missing-credit rules and cast format verified.")

Missing-credit rules and cast format verified.


## Keep One Row per Movie

After removing 37 exact duplicate rows, seven movie IDs still have two records each. Inspection showed the same cast names and crew members in each pair, although some crew entries have a different order. We keep the first record encountered for each movie ID so the choice is repeatable.

One limitation is that a movie with multiple credited directors can have a different first director when crew order changes. We retain the director from the selected source record and note this ambiguity.

In [30]:
# A valid movie ID must be present and greater than zero.
assert credits_parsed["id"].notna().all()
assert (credits_parsed["id"] > 0).all()

credits_unique = credits_parsed.drop_duplicates(
    subset=["id"],
    keep="first"
).copy()

assert credits_unique["id"].is_unique

print("Rows before resolving repeated IDs:", len(credits_parsed))
print("Rows after keeping one row per movie:", len(credits_unique))
print("Repeated IDs remaining:", credits_unique["id"].duplicated().sum())

Rows before resolving repeated IDs: 45439
Rows after keeping one row per movie: 45432
Repeated IDs remaining: 0


## Save the Cleaned Credits Dataset

The final table contains one row per movie with its `id`, director, top five cast members, cast size, and crew size. We store the cast names as JSON text, such as `["Tom Hanks", "Tim Allen"]`; no cast is stored as `[]`. A missing director is left blank in the CSV.

The processed CSV stays on this computer because the repository ignores the `data/` folder.

In [31]:
import json

cleaned_credits = credits_unique[
    ["id", "director", "top_five_cast", "cast_size", "crew_size"]
].copy()

# Convert every cast list to the same portable text format.
cleaned_credits["top_five_cast"] = cleaned_credits["top_five_cast"].apply(
    lambda names: json.dumps(names, ensure_ascii=False)
)

processed_path = project_root / "data" / "processed" / "credits.csv"
processed_path.parent.mkdir(parents=True, exist_ok=True)
cleaned_credits.to_csv(processed_path, index=False)

# Read the saved file back to verify the actual deliverable.
saved_credits = pd.read_csv(processed_path)

assert len(saved_credits) == 45432
assert saved_credits["id"].is_unique
assert list(saved_credits.columns) == [
    "id", "director", "top_five_cast", "cast_size", "crew_size"
]
assert saved_credits["top_five_cast"].apply(
    lambda value: isinstance(json.loads(value), list)
).all()

print("Saved file:", processed_path)
print("Saved rows:", len(saved_credits))
print("Duplicate IDs:", saved_credits["id"].duplicated().sum())
display(saved_credits.head(10))

Saved file: /Users/kevinphon/Desktop/DS-ML Project Fall Semester/css-dsml-fall2027/data/processed/credits.csv
Saved rows: 45432
Duplicate IDs: 0


,id,director,top_five_cast,cast_size,crew_size
0,862,John Lasseter,"[""Tom Hanks"", ""Tim Allen"", ""Don Rickles"", ""Jim...",13,106
1,8844,Joe Johnston,"[""Robin Williams"", ""Jonathan Hyde"", ""Kirsten D...",26,16
2,15602,Howard Deutch,"[""Walter Matthau"", ""Jack Lemmon"", ""Ann-Margret...",7,4
3,31357,Forest Whitaker,"[""Whitney Houston"", ""Angela Bassett"", ""Loretta...",10,10
4,11862,Charles Shyer,"[""Steve Martin"", ""Diane Keaton"", ""Martin Short...",12,7
5,949,Michael Mann,"[""Al Pacino"", ""Robert De Niro"", ""Val Kilmer"", ...",65,71
6,11860,Sydney Pollack,"[""Harrison Ford"", ""Julia Ormond"", ""Greg Kinnea...",57,53
7,45325,Peter Hewitt,"[""Jonathan Taylor Thomas"", ""Brad Renfro"", ""Rac...",7,4
8,9091,Peter Hyams,"[""Jean-Claude Van Damme"", ""Powers Boothe"", ""Do...",6,9
9,710,Martin Campbell,"[""Pierce Brosnan"", ""Sean Bean"", ""Izabella Scor...",20,46
